# Notebook 3: CLIP — Zero-Shot Transfer & Prompt Engineering Study on EuroSAT

This notebook explores **CLIP** (Contrastive Language-Image Pretraining) as a zero-shot and few-shot transfer learner on satellite imagery. CLIP represents a fundamentally different paradigm from the supervised transfer learning studied in Notebook 1 — it requires **no task-specific training examples** to classify images.

**What we study:**
1. How sensitive is CLIP's zero-shot accuracy to prompt wording? (The key experiment)
2. Does prompt ensembling reliably improve zero-shot performance?
3. What does the CLIP embedding space look like for EuroSAT?
4. How does a few-shot linear probe over CLIP features compare to full fine-tuning?
5. Can CLIP perform cross-modal text→image retrieval on satellite data?

**Reference**: Radford et al. (2021) *"Learning Transferable Visual Models From Natural Language Supervision"*

## Section 0: Imports & Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
from configs.clip_config import CLIP_MODEL_ID, EUROSAT_CLASSES, PROMPT_TEMPLATES, CLASS_DESCRIPTIONS, FEW_SHOT_K_VALUES, CLIPConfig
from src.clip.pipeline import run_clip_pipeline, extract_image_features, extract_text_features, build_prompts, zero_shot_classify, few_shot_linear_probe, retrieve_images
from src.utils.visualization import MODEL_COLORS
import json
from pathlib import Path
from PIL import Image

RESULTS_PATH = Path("../results/clip")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
Path("../results/figures").mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
print(f"CLIP model: {CLIP_MODEL_ID}")
print(f"Classes: {EUROSAT_CLASSES}")
print(f"Prompt templates ({len(PROMPT_TEMPLATES)}): {PROMPT_TEMPLATES}")

## Section 1: Introduction to CLIP

### What is CLIP and how does zero-shot classification work?

**CLIP** (Radford et al., 2021, OpenAI) is a dual-encoder model trained on 400 million (image, text) pairs scraped from the internet. The training objective is simple but powerful: for a batch of N image-text pairs, maximize the cosine similarity of the N correct pairs while minimizing similarity for the N²−N incorrect pairs (contrastive loss).

This produces two encoders — a visual encoder (ViT-B/32 by default) and a text encoder (Transformer) — whose output embeddings lie in a **shared semantic space**.

### Zero-shot classification:
For a new classification task with classes $\{c_1, ..., c_K\}$:
1. Encode each class name as text: `f_text("a photo of a {class_name}")`
2. Encode the query image: `f_image(image)`
3. Predict the class with highest cosine similarity

**No gradient updates, no labeled training data.** The model classifies based purely on semantic alignment learned during pretraining.

### Why satellite imagery is an interesting test case:
- **Large domain gap**: CLIP was trained on internet images (mostly photos, artwork, screenshots). Satellite imagery is top-down, lacks perspective, and has distinct visual properties.
- **Semantic ambiguity**: Class names like "AnnualCrop" or "HerbaceousVegetation" are technical jargon unlikely to appear in CLIP's training data.
- **Prompt matters**: Unlike ImageNet classes ("golden retriever", "keyboard"), satellite class descriptions require domain knowledge to phrase correctly.

This makes EuroSAT an excellent stress-test for CLIP's generalization — and for understanding when/why prompt engineering makes a difference.

## Section 2: Load CLIP and Extract EuroSAT Features

In [ ]:
# Load CLIP model and processor
print(f"Loading CLIP: {CLIP_MODEL_ID}")
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model.eval()

print(f"Visual encoder: {type(clip_model.vision_model).__name__}")
print(f"Text encoder:   {type(clip_model.text_model).__name__}")
print(f"Embedding dim:  {clip_model.config.projection_dim}")

# Parameter count
total_params = sum(p.numel() for p in clip_model.parameters())
print(f"Total params:   {total_params/1e6:.1f}M")

In [ ]:
# Load EuroSAT and extract features
dataset = load_dataset("blanchon/EuroSAT_RGB")
train_ds = dataset["train"]
test_ds  = dataset["test"] if "test" in dataset else dataset["validation"]

features_file = RESULTS_PATH / "eurosat_features.pt"

if features_file.exists():
    print("Loading cached features...")
    cached = torch.load(features_file, map_location="cpu")
    train_features = cached["train_features"]
    train_labels   = cached["train_labels"]
    test_features  = cached["test_features"]
    test_labels    = cached["test_labels"]
else:
    print("Extracting CLIP image features (this may take a few minutes)...")
    train_features, train_labels = extract_image_features(
        clip_model, clip_processor, train_ds, device=DEVICE, batch_size=64
    )
    test_features, test_labels = extract_image_features(
        clip_model, clip_processor, test_ds, device=DEVICE, batch_size=64
    )
    torch.save({
        "train_features": train_features, "train_labels": train_labels,
        "test_features": test_features,   "test_labels": test_labels,
    }, features_file)
    print("Features cached.")

print(f"\nFeature dimensionality: {test_features.shape[1]}")
print(f"Train features: {train_features.shape}  |  Test features: {test_features.shape}")
print(f"L2 norm (should be ~1.0): {test_features.norm(dim=-1).mean():.4f}")

## Section 3: Zero-Shot Classification — Prompt Sensitivity Study

### Why prompt wording matters

CLIP was trained on image-text pairs from the internet, where text descriptions follow natural language patterns. The model learned associations between visual concepts and how they are *typically described in text*. This means the choice of prompt can dramatically affect performance:

- `"a photo of {class}"` — generic, matches training distribution
- `"satellite image of {class}"` — accurate description but may be rare in training data
- `"aerial view of {class}"` — alternative framing; may activate different associations
- `"land use: {class}"` — domain-specific jargon; may or may not be familiar to CLIP
- `"{description}"` — full natural language description; tests if CLIP understands content

**The key insight**: CLIP's zero-shot performance is highly sensitive to how you phrase the class names. Understanding this sensitivity is critical for anyone deploying CLIP on domain-specific tasks.

In [ ]:
# Zero-shot classification for each prompt template
prompt_results_file = RESULTS_PATH / "prompt_sensitivity.json"

if prompt_results_file.exists():
    with open(prompt_results_file) as f:
        prompt_results = json.load(f)
else:
    prompt_results = {}
    for template in PROMPT_TEMPLATES:
        print(f"Testing template: '{template}'")
        prompts = build_prompts(EUROSAT_CLASSES, template)
        text_features = extract_text_features(clip_model, clip_processor, prompts, device=DEVICE)
        per_class_acc, overall_acc = zero_shot_classify(
            test_features, test_labels.numpy(), text_features
        )
        prompt_results[template] = {
            "overall_acc": overall_acc,
            "per_class_acc": per_class_acc
        }
        print(f"  → accuracy: {overall_acc*100:.2f}%")

    with open(prompt_results_file, "w") as f:
        json.dump(prompt_results, f, indent=2)

print("\nPrompt Sensitivity Results:")
sorted_templates = sorted(prompt_results.items(), key=lambda x: x[1]["overall_acc"], reverse=True)
for template, res in sorted_templates:
    print(f"  {res['overall_acc']*100:.2f}%  |  '{template}'")

In [ ]:
# Horizontal bar chart with accuracy per template
templates = [t for t, _ in sorted_templates]
accs = [prompt_results[t]["overall_acc"] * 100 for t in templates]

# Compute std across per-class accuracies as a proxy for consistency
stds = [np.std(list(prompt_results[t]["per_class_acc"].values())) * 100 for t in templates]

fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#2ecc71" if acc == max(accs) else ("#e74c3c" if acc == min(accs) else "#3498db")
          for acc in accs]

bars = ax.barh(range(len(templates)), accs, xerr=stds, color=colors,
               edgecolor="white", capsize=4, error_kw={"linewidth": 1.5, "ecolor": "#555"})

ax.set_yticks(range(len(templates)))
ax.set_yticklabels([f"'{t}'" for t in templates], fontsize=9)
ax.set_xlabel("Zero-Shot Accuracy (%)")
ax.set_title("CLIP Zero-Shot Accuracy per Prompt Template on EuroSAT\n(error bars = std across per-class accuracies)",
             fontsize=12, fontweight="bold")

for bar, acc, std in zip(bars, accs, stds):
    ax.text(acc + std + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{acc:.1f}%", va="center", fontsize=9, fontweight="bold")

ax.set_xlim(0, max(accs) + 8)
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Accuracy range annotation
acc_range = max(accs) - min(accs)
ax.text(0.02, 0.02, f"Accuracy range: {acc_range:.1f}% from prompt wording alone",
        transform=ax.transAxes, fontsize=9, style="italic",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff3cd", edgecolor="#ffc107"))

plt.tight_layout()
plt.savefig("../results/figures/clip_prompt_sensitivity.png", dpi=150)
plt.show()

print(f"\nKey Finding: CLIP zero-shot accuracy varies from {min(accs):.1f}% to {max(accs):.1f}%")
print(f"— a {acc_range:.1f}% gap driven purely by prompt wording.")
print(f"Best template:  '{templates[0]}'")
print(f"Worst template: '{templates[-1]}'")

## Section 4: Prompt Ensembling

### Averaging text embeddings over multiple prompt templates

Radford et al. (2021) found that using **80 diverse prompt templates** on ImageNet and averaging the resulting text embeddings improved zero-shot accuracy by ~3.5% over the single best template. The intuition: different phrasings activate different aspects of CLIP's learned associations, and their average is more robust.

Formally, for class $c$ with templates $\{t_1, ..., t_T\}$:
$$e_c^{\text{ensemble}} = \frac{1}{T} \sum_{i=1}^{T} f_{\text{text}}(t_i(c))$$

The ensemble embedding captures the centroid of all phrasings in the semantic space.

In [ ]:
# Compute ensemble accuracy
print("Computing ensemble accuracy over all templates...")

all_text_features = []
for template in PROMPT_TEMPLATES:
    prompts = build_prompts(EUROSAT_CLASSES, template)
    feats = extract_text_features(clip_model, clip_processor, prompts, device=DEVICE)
    all_text_features.append(feats)

# Average text embeddings (then renormalize)
ensemble_features = torch.stack(all_text_features).mean(dim=0)
ensemble_features = ensemble_features / ensemble_features.norm(dim=-1, keepdim=True)

_, ensemble_acc = zero_shot_classify(test_features, test_labels.numpy(), ensemble_features)
best_single_acc  = max(prompt_results[t]["overall_acc"] for t in PROMPT_TEMPLATES)
worst_single_acc = min(prompt_results[t]["overall_acc"] for t in PROMPT_TEMPLATES)

# Random single template (average of all single-template accuracies)
random_expected_acc = np.mean([prompt_results[t]["overall_acc"] for t in PROMPT_TEMPLATES])

print(f"\nResults:")
print(f"  Ensemble accuracy:            {ensemble_acc*100:.2f}%")
print(f"  Best single template:         {best_single_acc*100:.2f}%")
print(f"  Expected (random template):   {random_expected_acc*100:.2f}%")
print(f"  Worst single template:        {worst_single_acc*100:.2f}%")

In [ ]:
# Bar chart: ensemble vs best vs random vs worst
comparison = [
    ("Ensemble\n(all templates)",  ensemble_acc * 100,       "#2ecc71"),
    ("Best single\ntemplate",      best_single_acc * 100,    "#3498db"),
    ("Random template\n(expected)",random_expected_acc * 100,"#f39c12"),
    ("Worst single\ntemplate",     worst_single_acc * 100,   "#e74c3c"),
]

labels, vals, colors = zip(*comparison)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=1.2, width=0.55)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Zero-Shot Accuracy (%)")
ax.set_title("CLIP Prompt Strategy Comparison on EuroSAT",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, max(vals) + 8)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

gain = (ensemble_acc - random_expected_acc) * 100
ax.text(0.98, 0.95, f"Ensemble gain over\nrandom template: +{gain:.1f}%",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#d5f5e3", edgecolor="#2ecc71"))

plt.tight_layout()
plt.savefig("../results/figures/clip_prompt_ensembling.png", dpi=150)
plt.show()

## Section 5: CLIP Embedding Space Visualization

In [ ]:
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    print("UMAP not installed. Install with: pip install umap-learn")
    UMAP_AVAILABLE = False

if UMAP_AVAILABLE:
    # Sample 2000 test images for visualization
    n_viz = min(2000, len(test_features))
    viz_idx = np.random.choice(len(test_features), n_viz, replace=False)
    viz_features = test_features[viz_idx].numpy()
    viz_labels   = test_labels[viz_idx].numpy()

    print(f"Running UMAP on {n_viz} test samples (dim {viz_features.shape[1]} → 2)...")
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric="cosine", random_state=42, verbose=False)
    embedding_2d = reducer.fit_transform(viz_features)
    print("UMAP done.")

    # Scatter plot
    fig, ax = plt.subplots(figsize=(12, 9))
    palette = sns.color_palette("tab10", len(EUROSAT_CLASSES))

    for cls_idx, cls_name in enumerate(EUROSAT_CLASSES):
        mask = viz_labels == cls_idx
        ax.scatter(
            embedding_2d[mask, 0], embedding_2d[mask, 1],
            c=[palette[cls_idx]], label=cls_name,
            s=8, alpha=0.6, linewidths=0
        )

    ax.set_title("CLIP Embedding Space — EuroSAT Test Set (UMAP 2D)",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("UMAP Dimension 1")
    ax.set_ylabel("UMAP Dimension 2")
    ax.legend(title="Land-Use Class", bbox_to_anchor=(1.01, 1), loc="upper left",
              fontsize=9, markerscale=3)
    ax.axis("off")

    plt.tight_layout()
    plt.savefig("../results/figures/clip_umap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("UMAP visualization skipped. Install umap-learn to enable.")

## Section 6: Few-Shot Linear Probe

In [ ]:
# Few-shot linear probe at different k values
few_shot_file = RESULTS_PATH / "few_shot_results.json"

if few_shot_file.exists():
    with open(few_shot_file) as f:
        few_shot_results = json.load(f)
else:
    print("Running few-shot linear probe...")
    few_shot_results = {}
    for k in FEW_SHOT_K_VALUES:
        print(f"  k={k} shots per class ({k * len(EUROSAT_CLASSES)} total examples)...")
        acc = few_shot_linear_probe(
            train_features=train_features,
            train_labels=train_labels.numpy(),
            test_features=test_features,
            test_labels=test_labels.numpy(),
            k=k,
            num_classes=len(EUROSAT_CLASSES),
            n_trials=5  # average over multiple random subsets
        )
        few_shot_results[str(k)] = acc
        print(f"    → accuracy: {acc*100:.2f}%")

    with open(few_shot_file, "w") as f:
        json.dump(few_shot_results, f, indent=2)

print("\nFew-Shot Linear Probe Results:")
for k in FEW_SHOT_K_VALUES:
    acc = few_shot_results[str(k)]
    print(f"  k={k:3d} shots: {acc*100:.2f}%")

In [ ]:
# Accuracy vs k plot
k_values = FEW_SHOT_K_VALUES
accs_probe = [few_shot_results[str(k)] * 100 for k in k_values]

# Zero-shot ensemble as baseline
zs_baseline = ensemble_acc * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(k_values, accs_probe, "o-", color="#e74c3c", linewidth=2.5,
            markersize=8, markerfacecolor="white", markeredgewidth=2, label="CLIP Linear Probe")

ax.axhline(zs_baseline, color="#3498db", linestyle="--", linewidth=2,
           label=f"Zero-Shot Ensemble ({zs_baseline:.1f}%)")

# Find k where probe exceeds zero-shot
for k, acc in zip(k_values, accs_probe):
    if acc >= zs_baseline:
        ax.axvline(k, color="#2ecc71", linestyle=":", linewidth=1.5,
                   label=f"Beats zero-shot at k={k}")
        break

ax.set_xlabel("k (shots per class)", fontsize=11)
ax.set_ylabel("Test Accuracy (%)", fontsize=11)
ax.set_title("CLIP Few-Shot Linear Probe vs Zero-Shot Ensemble on EuroSAT",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Annotate data points
for k, acc in zip(k_values, accs_probe):
    ax.annotate(f"{acc:.1f}%", (k, acc), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../results/figures/clip_few_shot_curve.png", dpi=150)
plt.show()

## Section 7: Cross-Modal Retrieval — Text → Image

### Shared embedding space enables natural language image search

Because CLIP encodes both images and text into a **shared semantic space**, you can use natural language queries to retrieve images — without any training on retrieval tasks. This is **zero-shot image retrieval** via embedding similarity:

1. Embed the text query: `q = f_text("a satellite image of a river")`
2. Compute cosine similarity with all image embeddings in the index
3. Return the top-k images by similarity score

This has direct practical applications: searching large remote sensing archives by natural language description, rapid annotation assistance, and anomaly detection.

We build a cached retrieval index of **100 images per class = 1,000 total** from EuroSAT.

In [ ]:
# Build retrieval index: 100 images per class = 1000 total
N_PER_CLASS = 100
index_file  = RESULTS_PATH / "retrieval_index.pt"

if index_file.exists():
    index_data = torch.load(index_file, map_location="cpu")
    index_features = index_data["features"]
    index_labels   = index_data["labels"]
    index_images   = index_data["images"]  # PIL images stored as numpy arrays
    print(f"Loaded retrieval index: {len(index_features)} images")
else:
    print("Building retrieval index...")
    index_imgs, index_lbls = [], []
    from collections import defaultdict
    per_class = defaultdict(list)

    for sample in train_ds:
        lbl = sample["label"]
        if len(per_class[lbl]) < N_PER_CLASS:
            img = sample["image"]
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img))
            per_class[lbl].append((img, lbl))
        if all(len(v) == N_PER_CLASS for v in per_class.values()):
            break

    for lbl, pairs in sorted(per_class.items()):
        for img, l in pairs:
            index_imgs.append(img)
            index_lbls.append(l)

    index_features, _ = extract_image_features(
        clip_model, clip_processor,
        [{"image": img, "label": lbl} for img, lbl in zip(index_imgs, index_lbls)],
        device=DEVICE, batch_size=64
    )
    index_labels = torch.tensor(index_lbls)
    index_images = [np.array(img.resize((64, 64))) for img in index_imgs]

    torch.save({
        "features": index_features, "labels": index_labels, "images": index_images
    }, index_file)
    print(f"Index built: {len(index_features)} images ({N_PER_CLASS} per class)")

In [ ]:
# Cross-modal retrieval demonstration
queries = [
    "a satellite image of a river",
    "urban areas with buildings",
    "green forested areas",
    "large bodies of water",
    "industrial land",
]

TOP_K = 5

fig, axes = plt.subplots(len(queries), TOP_K + 1, figsize=(15, 3 * len(queries)))
fig.suptitle("CLIP Cross-Modal Text → Image Retrieval on EuroSAT",
             fontsize=14, fontweight="bold")

for row, query in enumerate(queries):
    # Encode query
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        text_feat = clip_model.get_text_features(**inputs)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        text_feat = text_feat.cpu()

    # Compute similarities
    sims = (index_features @ text_feat.T).squeeze(1)
    top_k_idx = sims.topk(TOP_K).indices.tolist()
    top_k_sims = sims[top_k_idx].tolist()

    # Query text cell (leftmost)
    ax_text = axes[row][0]
    ax_text.text(0.5, 0.5, f"Query:\n'{query}'", ha="center", va="center",
                 fontsize=9, wrap=True, fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="#ebf5fb", edgecolor="#3498db"))
    ax_text.axis("off")

    # Top-k retrieved images
    for col, (img_idx, sim) in enumerate(zip(top_k_idx, top_k_sims)):
        ax = axes[row][col + 1]
        img_arr = index_images[img_idx]
        if isinstance(img_arr, np.ndarray):
            ax.imshow(img_arr)
        else:
            ax.imshow(np.array(img_arr))
        cls_name = EUROSAT_CLASSES[index_labels[img_idx].item()]
        ax.set_title(f"#{col+1}: {cls_name}\nsim={sim:.3f}", fontsize=7)
        ax.axis("off")

plt.tight_layout()
plt.savefig("../results/figures/clip_retrieval.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 8: Per-Class Zero-Shot Analysis

In [ ]:
# Per-class zero-shot accuracy using ensemble
from sklearn.metrics import accuracy_score

# Get ensemble predictions
sims = test_features @ ensemble_features.T
ensemble_preds = sims.argmax(dim=-1).numpy()
true_labels    = test_labels.numpy()

per_class_acc = {}
for cls_idx, cls_name in enumerate(EUROSAT_CLASSES):
    mask = true_labels == cls_idx
    if mask.sum() > 0:
        per_class_acc[cls_name] = (ensemble_preds[mask] == cls_idx).mean()

# Sorted bar chart
sorted_classes = sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True)
cls_names, cls_accs = zip(*sorted_classes)
cls_accs_pct = [a * 100 for a in cls_accs]

palette = ["#2ecc71" if a >= 70 else ("#f39c12" if a >= 50 else "#e74c3c")
           for a in cls_accs_pct]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(cls_names, cls_accs_pct, color=palette, edgecolor="white")
ax.axhline(ensemble_acc * 100, color="black", linestyle="--", linewidth=1.5,
           label=f"Overall: {ensemble_acc*100:.1f}%")

for bar, acc in zip(bars, cls_accs_pct):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{acc:.0f}%", ha="center", va="bottom", fontsize=9)

ax.set_title("CLIP Zero-Shot Accuracy per Class — Ensemble Prompts on EuroSAT",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 110)
ax.tick_params(axis="x", rotation=30)
ax.legend(fontsize=10)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ecc71", label="Good (≥70%)"),
    Patch(facecolor="#f39c12", label="Moderate (50–70%)"),
    Patch(facecolor="#e74c3c", label="Poor (<50%)"),
]
ax.legend(handles=legend_elements + [plt.Line2D([0], [0], color="black", linestyle="--",
          label=f"Overall: {ensemble_acc*100:.1f}%")], fontsize=9)

plt.tight_layout()
plt.savefig("../results/figures/clip_per_class_accuracy.png", dpi=150)
plt.show()

print("\nPer-class analysis:")
print(f"\nStrong classes (CLIP handles well):")
for cls, acc in sorted_classes[:3]:
    desc = CLASS_DESCRIPTIONS.get(cls, "N/A")
    print(f"  {cls:<25} {acc*100:.1f}% — Description: '{desc[:60]}'")

print(f"\nWeak classes (CLIP struggles):")
for cls, acc in sorted_classes[-3:]:
    desc = CLASS_DESCRIPTIONS.get(cls, "N/A")
    print(f"  {cls:<25} {acc*100:.1f}% — Description: '{desc[:60]}'")